# Data Structures — CO1

**2310214L.CO.1** — *Develop applications using arrays.* **[L3]**

This notebook covers **CO1 only**, in depth. The other outcomes are in
[`subjects/01-data-structures.md`](subjects/01-data-structures.md).

---

## What an array actually is

An array is one continuous block of memory holding items of the same size, one after
another. Nothing else.

That single fact gives it the property everything below depends on: **the position of an
item is its name**. If you want item number 5, you do not search for it — you calculate
where it must be and go straight there.

```
address of item i  =  start of the block  +  ( i × size of one item )
```

That is one multiplication and one addition, no matter how large the array is. Reading
item 5 costs the same as reading item 500. A linked list cannot do this: to reach item 5
it must start at the front and follow five links.

So the rule we applied throughout the project is simple:

> **If a thing is identified by its position, use an array. If it is identified by
> something else, or has no fixed count, use something else.**

Everything in this notebook is an application of that one rule.

---

## Where we used arrays, and why

Four places in the project. Each one passes the test above.

| What | Where | What the index means |
|---|---|---|
| The register file | `src/core/RegisterFile.h` | the register number |
| Instruction memory | `src/core/CPU.cpp` | the program counter |
| The instruction fetch queue | `src/ds/CircularQueue.h` | position in the waiting line |
| The screen buffer | `src/ui/Canvas.cpp` | the pixel's position |

---

## 1 · The register file

![the register file as an array](diagrams/ds-arrays.png)

A processor has a small, fixed set of registers — ours has eight, named R0 to R7. They
never change in number while the machine runs, and an instruction always names one by its
number.

**What this does.** Declares all eight registers as one array. The register number is the
position in that array, so `gp_[5]` *is* R5 — there is nothing to look up.

```cpp
Register gp_[NUM_GP_REGISTERS];   // R0..R7 — a plain fixed array
```
<sub>src/core/RegisterFile.h:51</sub>

**What this does.** Reads one register. The index is checked first, because a bad index in
an array does not fail safely — it quietly reads whatever memory happens to sit there.

```cpp
Word RegisterFile::readGP(int index) {
    if (index < 0 || index >= NUM_GP_REGISTERS)
        throw InvalidRegisterException("read index out of range");
    return gp_[index].read();
}
```
<sub>src/core/RegisterFile.cpp:32</sub>

**Why an array here.** Every instruction reads or writes a register, and instructions run
constantly. If reading R5 meant walking a list, the cost would be paid millions of times
over. With an array it is a single step.

**What the array costs us.** The size is fixed at compile time. We cannot decide at
run-time to have twelve registers. For a processor that is not a limitation but an honest
reflection of the hardware — real processors have a fixed register count too, for exactly
the same reason.

---

## 2 · Instruction memory

The assembled program is held as an array of instruction pointers, and the program counter
is the index into it.

**What this does.** Fetches whichever instruction the program counter is pointing at.
Because the program counter *is* the index, "go to the next instruction" is just adding
one to a number.

```cpp
Instruction* CPU::instructionAt(unsigned int address) const {
    if (address < programBase_) return 0;
    unsigned int idx = address - programBase_;
    if (idx >= programSize_) return 0;
    return program_[idx];
}
```
<sub>src/core/CPU.cpp:61</sub>

This is worth pausing on, because it explains something about real machines. A jump
instruction does not "find" a line — it writes a number into the program counter. The
whole idea of jumping only works because instructions live at *positions*, and positions
are what an array gives you.

---

## 3 · The circular queue — an array that never runs out

Instructions wait in a line before being executed. A queue is the natural structure, but a
plain array queue has a problem: every time you remove from the front, everything else has
to shift up one place, or the free space at the front is wasted and the queue eventually
runs off the end of the array.

The fix is to stop moving the data and move the *index* instead, wrapping it around.

**What this does.** Adds an item at the back. The `%` is the entire trick — when the
position reaches the end of the array it wraps back to zero, so the same block of memory
is reused forever.

```cpp
void enqueue(const T& value) {
    if (count_ == capacity_)
        throw core::IndexOutOfRangeException("CircularQueue::enqueue on full queue");
    buf_[(front_ + count_) % capacity_] = value;
    ++count_;
}
```
<sub>src/ds/CircularQueue.h:52</sub>

**What this does.** Takes the item from the front. Nothing is shifted; the front marker
simply moves along, wrapping in the same way.

```cpp
T dequeue() {
    if (count_ == 0)
        throw core::IndexOutOfRangeException("CircularQueue::dequeue on empty queue");
    T value = buf_[front_];
    front_  = (front_ + 1) % capacity_;
    --count_;
    return value;
}
```
<sub>src/ds/CircularQueue.h:59</sub>

**Worked example.** Capacity 4, and the queue currently holds three items starting at
position 2:

```
positions:   0     1     2     3
contents:  [ - ] [ - ] [ A ] [ B ]      front_ = 2, count_ = 2
```

Add C. Its position is `(2 + 2) % 4` = `0` — it wraps to the front of the array, even
though it is at the back of the queue:

```
positions:   0     1     2     3
contents:  [ C ] [ - ] [ A ] [ B ]      front_ = 2, count_ = 3
```

The array is never reallocated, nothing is ever copied, and the queue behaves as if it had
no end. This is why real processor fetch buffers are built the same way.

---

## 4 · The screen buffer — a 2D picture inside a 1D array

The display is a grid of pixels, but memory is not a grid. It is one long line. So the two
dimensions have to be folded into one.

**What this does.** Turns an (x, y) position into a single array index. Everything the
project draws goes through this one function.

```cpp
void Canvas::setPixel(int x, int y, Pixel p) {
    if (inBounds(x, y)) buffer_[y * width_ + x] = p;
}
```
<sub>src/ui/Canvas.cpp:41</sub>

**Why `y * width_ + x` works.** Rows are stored one after another. To reach row `y` you
skip `y` whole rows, each `width_` long — that is `y * width_`. Then you step `x` places
into that row.

```
width = 5

row 0:  [ 0] [ 1] [ 2] [ 3] [ 4]
row 1:  [ 5] [ 6] [ 7] [ 8] [ 9]
row 2:  [10] [11] [12] [13] [14]

pixel (x=3, y=2)  ->  2 * 5 + 3  =  index 13
```

**What this does.** Allocates the buffer. The size is width times height, in one block.

```cpp
buffer_ = new Pixel[width_ * height_];
```
<sub>src/ui/Canvas.cpp:10</sub>

Every drawing routine in the project — the lines, the circles, the fills — ends up calling
`setPixel`, so this one index calculation sits underneath the entire display.

---

## The one thing an array will not forgive

An array does not check anything for you. `buffer_[999999]` compiles happily and reads
whatever memory is at that address, which is why we guard every access:

```cpp
bool Canvas::inBounds(int x, int y) const {
    return x >= 0 && x < width_ && y >= 0 && y < height_;
}
```
<sub>src/ui/Canvas.cpp:33</sub>

The same pattern appears in `RegisterFile::readGP`, in `CircularQueue::at`, and everywhere
else an index arrives from outside. It is the price of the speed: the array gives you
instant access and trusts you completely, so the checking has to be yours.

---

## How to explain CO1 in one minute

1. An array stores items back to back, so **position is identity** — item 5 is found by
   arithmetic, not by searching.
2. We used arrays in the four places where something genuinely is identified by position:
   registers by number, instructions by program counter, queue slots by order, pixels by
   coordinate.
3. The circular queue shows the clever case — wrapping the *index* with `%` instead of
   moving the *data*, so a fixed array behaves as though it had no end.
4. The screen buffer shows the folding case — a 2D picture living in a 1D array through
   `y * width + x`.
5. And the cost: an array checks nothing, so every index that comes from outside is
   guarded before use.